# 08 — Gel de la release (manifest SHA-256, lecture seule)

```text
Gel de la release : manifest SHA-256 de TOUS les artefacts, lecture seule, archive partageable.

python 08_freeze.py            # exige QC sans échec, écrit manifest.json, passe les fichiers en lecture seule
python 08_freeze.py --verify   # (autres groupes) vérifie que leurs fichiers == ceux de la release
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [1]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [2]:
import argparse, os, stat, sys, tarfile
from common import env_info, get_logger, load_config, paths, read_json, sha256_file, write_json
log = get_logger("freeze")
SHARE_DIRS = ["splits", "tables", "embeddings", "index", "eda", "qc", "audit"]

### `main()`

In [3]:
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--verify", action="store_true"); ap.add_argument("--no-tar", action="store_true")
    args = ap.parse_args()
    cfg = load_config(); p = paths(cfg); art = p["art"]
    if args.verify:
        from rag_data import Release
        bad = Release(art, cfg["primary_encoder"]).verify()
        log.info("release identique à la référence" if not bad else f"DIFFÉRENCES: {bad[:10]} ({len(bad)} fichiers)")
        sys.exit(1 if bad else 0)
    if not (p["qc"] / "qc_report.json").exists():
        sys.exit("Rapport QC introuvable : exécute d'abord le notebook 07_qc_leakage.")
    qc = read_json(p["qc"] / "qc_report.json")
    if not qc["passed"]:
        fails = [f'{c["check"]} -> {c["detail"]}' for c in qc["checks"] if c["status"] == "FAIL"]
        sys.exit("GEL REFUSÉ : le QC (07) a des échecs. Corrige-les puis relance 07 et 08 :\n - " + "\n - ".join(fails))
    files = {}
    for f in sorted(art.rglob("*")):
        if f.is_file() and f.name != "manifest.json" and not f.name.endswith(".tar.gz"):
            files[str(f.relative_to(art))] = {"sha256": sha256_file(f), "bytes": f.stat().st_size}
    man = {"release_version": cfg["release_version"], "seed": cfg["seed"], "sizes": cfg["sizes"], "primary_encoder": cfg["primary_encoder"],
           "encoders": list(cfg["encoders"]), "env": env_info(), "n_files": len(files), "files": files, "config": cfg}
    manifest_path = art / "manifest.json"
    if manifest_path.exists():
        # Autoriser la r??g??n??ration du manifeste lors d'un nouveau gel.
        os.chmod(manifest_path, stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)
    write_json(manifest_path, man)
    for f in art.rglob("*"):
        if f.is_file():
            os.chmod(f, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
    log.info("release %s figée : %d fichiers, lecture seule", cfg["release_version"], len(files))
    if not args.no_tar:                                      # archive légère (sans les JPEG, régénérables) pour partage
        tgz = art.parent / f"release_{cfg['release_version']}.tar.gz"
        with tarfile.open(tgz, "w:gz") as t:
            for d in SHARE_DIRS:
                if (art / d).exists():
                    t.add(art / d, arcname=f"{art.name}/{d}")
            t.add(art / "manifest.json", arcname=f"{art.name}/manifest.json")
        log.info("archive : %s (%.1f Mo)", tgz, tgz.stat().st_size / 1e6)

## Lancer l'étape

In [4]:
import sys
sys.argv = ["notebook"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

01:05:30 | INFO | Loading faiss with AVX2 support.
01:05:30 | INFO | Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
01:05:30 | INFO | Loading faiss.
01:05:30 | INFO | Successfully loaded faiss.
01:06:21 | INFO | release g1-v1.0 figée : 10254 fichiers, lecture seule
01:07:01 | INFO | archive : C:\Users\Cerine\Desktop\image\rag_preproc\release_g1-v1.0.tar.gz (374.9 Mo)


## Résultats

In [5]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Manifest

In [6]:
mp = P["art"] / "manifest.json"
if not mp.exists():
    print("Pas de manifest : le gel a été refusé (lis le message de la cellule « Lancer l'étape ») ou 07 n'a pas été exécuté.")
else:
    man = json.load(open(mp))
    print(man["release_version"], "| fichiers :", man["n_files"], "| encodeur principal :", man["primary_encoder"], "| seed :", man["seed"])
    display(pd.DataFrame(man["files"]).T.head(8))

g1-v1.0 | fichiers : 10254 | encodeur principal : clip-b32 | seed : 42


,sha256,bytes
audit\dedup_repair.json,c8dea38330a2b64d181e3f75c92208f25e1f9bd55469fb...,317
audit\detector_info.json,3db4d64cfa398c9aea3b3a84e584a97e0d98cc2a18c966...,248
audit\image_audit.json,f81f58b1ab11874cc01be8394252c6d99b351b48ab4a89...,397
audit\invalid_images.json,bfb56456d135bd518bf8cf15c6ad6ac4917e57ee84f50d...,35
audit\metadata_info.json,4fae11259bcf9722df672488f7f30dfeb375faf700b141...,784
audit\preproc_config_selection.json,9dda6cb57105a5bd5c9c51fc00e75901b2368eeed599c7...,2032
eda\01_top50_classes.png,8f4be79354f1704e570361049a547c6c8a9ebcda8d0f1d...,81049
eda\02_scene_complexity.png,cf0379f188c5198ee3a0d985abd85f68b940e2fafbc2f8...,63454


### Vérification d'intégrité (à lancer par les autres groupes)

In [7]:
from rag_data import Release
if not (P["art"] / "manifest.json").exists():
    print("Pas de manifest à vérifier (gel non effectué).")
else:
    bad = Release(P["art"], cfg["primary_encoder"]).verify()
    print("release identique à la référence ✔" if not bad else f"DIFFÉRENCES : {bad[:10]}")

release identique à la référence ✔
